In [57]:
import os
import re
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.models import Model

In [66]:
# Loading the data



def load_data(path, num_samples):
  eng_txt , spa_txt = [], []
  with open (path , 'r' , encoding= 'utf-8') as f:
    lines = f.read().split('\n')

  for line in lines [:min(num_samples, len(lines)-1)]:
    if not line.strip():
      continue

    parts = line.split('\t')
    eng, spa = parts[0],parts[1]
    eng_txt.append(eng)
    spa_txt.append(spa)

  return eng_txt, spa_txt

In [67]:
# Cleaning the data
def clean_text(text):
  text  = text.lower()
  text = re.sub(r"[^a-zA-ZáéíóúñÁÉÍÓÚÑ¿¡ ]+" , "", text)

  return text.strip()

def preprocess(eng_txt, spa_txt):
    eng_clean = [clean_text(t) for t in eng_txt]
    spa_clean = ["<start> "+clean_text(t)+ " <end>" for t in spa_txt]
    return eng_clean , spa_clean


In [68]:
# Tokenizer -> converting the data into integer
def tokenize(texts):
  tokenizer = Tokenizer(filters="")
  tokenizer.fit_on_texts(texts)
  sequences = tokenizer.texts_to_sequences(texts)
  padded = pad_sequences(sequences, padding ="post")

  return padded, tokenizer

In [69]:
# Creating the Encoder and Decoder Model
def build_training_model(eng_vocab_size, spa_vocab_size, embedding_dim = 256, latent_dim = 256):

  # Encoder
  encoder_inputs = Input(shape = (None,), name= "encoder_inputs")
  enc_emb = Embedding(eng_vocab_size, embedding_dim, mask_zero=True)(encoder_inputs)
  encoder_lstm = LSTM(latent_dim, return_state= True, name= 'encoder_lstm')
  _,state_h, state_c = encoder_lstm(enc_emb)
  encoder_states = [state_h, state_c] #context Vector passes to Decoder

   # Decoder
  decoder_inputs = Input(shape = (None,), name= "decoder_inputs")
  dec_embedding_layer = Embedding(spa_vocab_size,embedding_dim, mask_zero=True)
  dec_emb = dec_embedding_layer(decoder_inputs)
  decoder_lstm = LSTM(latent_dim, return_sequences= True, return_state= True, name="decoder_lstm")
  decoder_outputs,_,_ = decoder_lstm(dec_emb, initial_state=encoder_states)
  decoder_dense = Dense(spa_vocab_size, activation="softmax", name="decoder_dense")
  decoder_outputs = decoder_dense(decoder_outputs)
  model = Model([encoder_inputs, decoder_inputs] , decoder_outputs)

  return model, encoder_inputs, encoder_states, decoder_inputs, dec_embedding_layer,decoder_lstm, decoder_dense





In [70]:
# Traing array
def make_decoder_io(spa_padded):
  decoder_input_data = spa_padded[:, :-1]  #keep all row but drop last column
  decoder_target_data = spa_padded[:,1:]  #keep all row but drop first column
  return decoder_input_data, decoder_target_data

In [71]:
# Building the interface , inorder there will be no spanish while predictiong for english

def build_interface_models (encoder_inputs, encoder_states,
                            decoder_inputs,
                            dec_embedding_layer,
                            decoder_lstm,
                            decoder_dense, latent_dim=256):
  encoder_model = Model(encoder_inputs, encoder_states)
  decoder_state_input_h = Input(shape=(latent_dim,))
  decoder_state_input_c = Input(shape = (latent_dim,))
  decoder_state_inputs = [decoder_state_input_h, decoder_state_input_c]

  dec_emb2 = dec_embedding_layer(decoder_inputs)
  decoder_outputs2, state_h2, state_c2 = decoder_lstm(dec_emb2,
                                                      initial_state = decoder_state_inputs)
  decoder_states2 = [state_h2, state_c2]
  decoder_outputs2 = decoder_dense(decoder_outputs2)
  decoder_model = Model(
      [decoder_inputs] +decoder_state_inputs,
      [decoder_outputs2] + decoder_states2
  )
  return encoder_model, decoder_model

def translate_senteces(input_text, encoder_model, decoder_model,eng_tokenizer, spa_tokenizer,max_spa_len):
    input_seq = clean_text(input_text)
    seq = eng_tokenizer.texts_to_sequences([input_seq])
    seq = pad_sequences(seq, maxlen = encoder_model.input_shape[1], padding="post")

    states_value = encoder_model.predict(seq, verbose= 0)
    target_seq = np.array([[spa_tokenizer.word_index["<start>"]]])
    stop = False
    translated = []
    while not stop:
      output_tokens, h, c = decoder_model.predict([target_seq] + states_value, verbose = 0)
      sampled_token_index = np.argmax(output_tokens[0,-1,:])
      sampled_word = spa_tokenizer.index_word.get(sampled_token_index,"")

      if sampled_word == "<end>" or len(translated) >= max_spa_len:
        stop = True
      else:
          translated.append(sampled_word)

      target_seq = np.array([[sampled_token_index]])
      states_value = [h,c]
    return " ".join(translated)

In [88]:
# lets finish it up

if __name__ == "__main__":
  print("Loading and cleaning the data")
  eng_raw, spa_raw = load_data("/content/spa.txt", 20000)
  eng_texts , spa_texts = preprocess(eng_raw, spa_raw)

  print("Tokenizing")
  eng_padded, eng_tokenizer= tokenize(eng_texts)
  spa_padded, spa_tokenizer = tokenize(spa_texts)

  eng_vocab_size = len(eng_tokenizer.word_index) + 1
  spa_vocab_size = len(spa_tokenizer.word_index) + 1

  print(f"English vocab_size : {eng_vocab_size} , spanish_vocab_size: {spa_vocab_size}")

  print("Decoder Input/Target arrays are on the way ")
  decoder_input_data, decoder_target_data = make_decoder_io(spa_padded)
  decoder_target_data = np.expand_dims(decoder_target_data, -1)

  print("Building the training Model ")
  (model, encoder_inputs, encoder_states,decoder_inputs, dec_embedding_layer, decoder_lstm,decoder_dense) = build_training_model(eng_vocab_size, spa_vocab_size)
  model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")
  model.summary()

  print("Training....")

  model.fit([eng_padded, decoder_input_data],
            decoder_target_data,batch_size =64, epochs = 100,
            validation_split = 0.2)

  print("Building interface model ")
  encoder_model , decoder_model = build_interface_models(encoder_inputs, encoder_states,decoder_inputs, dec_embedding_layer,decoder_lstm,decoder_dense)
  max_spa_len = spa_padded.shape[1]



Loading and cleaning the data
Tokenizing
English vocab_size : 3709 , spanish_vocab_size: 7916
Decoder Input/Target arrays are on the way 
Building the training Model 


Model: "functional_24"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_32        │ (None, None, 256) │    949,504 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_36        │ (None, None)      │          0 │ encoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_33        │ (None, None, 256) │  2,026,496 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 256),     │    525,312 │ embedding_32[0][… │
│                     │ (None, 256),      │            │ not_equal_36[0][… │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, None,     │    525,312 │ embedding_33[0][… │
│                     │ 256), (None,      │            │ encoder_lstm[0][… │
│                     │ 256), (None,      │            │ encoder_lstm[0][… │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_dense       │ (None, None,      │  2,034,412 │ decoder_lstm[0][… │
│ (Dense)             │ 7916)             │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 6,061,036 (23.12 MB)

 Trainable params: 6,061,036 (23.12 MB)

 Non-trainable params: 0 (0.00 B)

Training....
Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 9s 28ms/step - loss: 4.6380 - val_loss: 4.2952
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 7s 28ms/step - loss: 3.5755 - val_loss: 3.9163
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 7s 27ms/step - loss: 3.0723 - val_loss: 3.5905
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 7s 28ms/step - loss: 2.6684 - val_loss: 3.3731
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 7s 27ms/step - loss: 2.3177 - val_loss: 3.2240
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 7s 28ms/step - loss: 2.0102 - val_loss: 3.1060
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 7s 28ms/step - loss: 1.7458 - val_loss: 3.0204
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 7s 29ms/step - loss: 1.5141 - val_loss: 2.9468
Epoch 9/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 8s 31ms/step - loss: 1.3113 - val_loss: 2.9251
Epoch 10/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 7s 28ms/step - loss: 1.1357 - val_loss: 2.9157
Epoch 11/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 7s 28ms/step - loss: 0.9839 - val_loss: 2.9050
Epoch 12/100
250/

In [92]:
# Translating
text_senteces = "the tasty food "
translation = translate_senteces(text_senteces,encoder_model, decoder_model,eng_tokenizer, spa_tokenizer,max_spa_len)
print(translation)

la comida está buena
